In [1]:
# Import packages
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
import pandas as pd
import earthpy.plot as ep
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.svm import SVR
#from pyearth import Earth
from keras import Model, Sequential
from keras.layers import Dense, Dropout, Input, BatchNormalization
from keras.callbacks import EarlyStopping

ModuleNotFoundError: No module named 'numpy'

In [3]:
# Parameter
FEATURES = [ 'logBLUE', 'logGREEN' ]
LABEL = 'agc'

# Sample parameter
SAMPLE = 'Mandalika_2018_October_v2.csv'

# Data parameter
DATA = 'Mandalika_2019_2023_v2.csv'
YEARS = [ 2019, 2020, 2021, 2022, 2023 ]
COLORS = [ 'purple', 'blue', 'green', 'orange', 'red' ]
MONTHS = [ 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12 ]

In [29]:
# Read sample data
sample_df = pd.read_csv(SAMPLE)

# Split into train and test
train, test = train_test_split(sample_df, train_size=0.8)
# print('Train', train)
# print('Test', test)

# Input and output
input_train = train[ FEATURES ]
output_train = train[ LABEL ]
input_test = test[ FEATURES ]
output_test = test[ LABEL ]

# Merge again for export between train adn test data
train['sample'] = 'train'
test['sample'] = 'test'
combined =  pd.concat([train, test])
combined = combined[FEATURES + ['logRED'] + [LABEL] + ['x', 'y', 'sample']]
combined.to_excel('sample_Mandalika_Oktober_2018_v2.xlsx')

In [4]:
# Monthly data
monthly_df = pd.read_csv(DATA)
monthly_df

,logBLUE,logGREEN,logRED,year,month
0,6.705639,6.677083,6.282267,2019,1
1,6.536692,6.472346,6.079933,2019,1
2,6.897705,6.857514,6.634633,2019,1
3,7.069874,7.091742,6.731018,2019,1
4,6.779922,6.739337,6.386879,2019,1
...,...,...,...,...,...
4098315,7.867871,7.898040,7.801391,2023,12
4098316,7.619724,7.631432,7.635304,2023,12
4098317,8.283999,8.206311,8.225235,2023,12
4098318,8.133294,8.206311,8.092545,2023,12


In [5]:
# Step wise
def stepwise(logGREEN):
	return 155.34 - (21.383 * logGREEN)

In [16]:
# Normal regression
linear = LinearRegression()
linear.fit(input_train, output_train)
score_linear = round(
    np.corrcoef(output_train, linear.predict(input_train))[0, 1] ** 2, 3
)
print(f'Linear R2: {score_linear}')

print(linear.coef_)
print(linear.intercept_)

Linear R2: 0.112
[-16.01400772   1.36520166]
138.09126100395625


In [7]:
# SVM ORI
svm_ori = SVR()
svm_ori.fit(input_train, output_train)
svm_ori_score = round(
    np.corrcoef(output_train, svm_ori.predict(input_train))[0, 1] ** 2, 3
)
print(f'SVM R2 Default: {svm_ori_score}')

""" # Search for best model
svm_param = {
	'C': [0.1, 1, 10, 100],
	'epsilon': [0.1, 0.2, 0.5, 1.0]
}
svm_grid = GridSearchCV(SVR(), svm_param)
svm_grid.fit(input_train, output_train)
print('Best SVR parameter', svm_grid.best_params_)
print('Best SVR score', svm_grid.best_score_)
svm = svm_grid.best_estimator_

# Score of SVM
score_svm = svm.score(input_train, output_train) """

SVM R2 Default: 0.103


" # Search for best model\nsvm_param = {\n\t'C': [0.1, 1, 10, 100],\n\t'epsilon': [0.1, 0.2, 0.5, 1.0]\n}\nsvm_grid = GridSearchCV(SVR(), svm_param)\nsvm_grid.fit(input_train, output_train)\nprint('Best SVR parameter', svm_grid.best_params_)\nprint('Best SVR score', svm_grid.best_score_)\nsvm = svm_grid.best_estimator_\n\n# Score of SVM\nscore_svm = svm.score(input_train, output_train) "

In [8]:
# RF ORI
rf_ori = RandomForestRegressor()
rf_ori.fit(input_train, output_train)
rf_ori_score = round(
    np.corrcoef(output_train, rf_ori.predict(input_train))[0, 1] ** 2, 3
)
print(f'RF R2 Default: {rf_ori_score}')

""" # Search for best model
rf_param = {
	'max_depth': [ 20, 30, 40 ],
	'min_samples_split': [ 2, 5, 10 ],
	'min_samples_leaf': [ 1, 2, 4 ],
}
rf_grid = GridSearchCV(RandomForestRegressor(), rf_param)
rf_grid.fit(input_train, output_train)
print('Best RF parameter', rf_grid.best_params_)
print('Best RF score', rf_grid.best_score_)
rf = rf_grid.best_estimator_

# Score RF
score_rf = rf.score(input_train, output_train) """

RF R2 Default: 0.796


" # Search for best model\nrf_param = {\n\t'max_depth': [ 20, 30, 40 ],\n\t'min_samples_split': [ 2, 5, 10 ],\n\t'min_samples_leaf': [ 1, 2, 4 ],\n}\nrf_grid = GridSearchCV(RandomForestRegressor(), rf_param)\nrf_grid.fit(input_train, output_train)\nprint('Best RF parameter', rf_grid.best_params_)\nprint('Best RF score', rf_grid.best_score_)\nrf = rf_grid.best_estimator_\n\n# Score RF\nscore_rf = rf.score(input_train, output_train) "

In [9]:
# XGB ORI
xgb_ori = xgb.XGBRegressor()
xgb_ori.fit(input_train, output_train)
xgb_ori_score = round(
    np.corrcoef(output_train, xgb_ori.predict(input_train))[0, 1] ** 2, 3
)
print(f'XGB R2 Default: {xgb_ori_score}')

""" # Search for best model
xgb_param = {
	'max_depth': [3, 4, 5],
	'min_child_weight': [1, 3, 5],
	'subsample': [0.8, 0.9, 1.0],
	'colsample_bytree': [0.8, 0.9, 1.0],
	'gamma': [0, 0.1, 0.2],
	'reg_alpha': [0, 0.1, 0.2],
	'reg_lambda': [0, 0.1, 0.2]
}
xgb_grid = GridSearchCV(xgb.XGBRegressor(), xgb_param)
xgb_grid.fit(input_train, output_train)
print('Best XGB parameter', xgb_grid.best_params_)
print('Best XGB score', xgb_grid.best_score_)
xgboost = xgb_grid.best_estimator_

score_xgboost = xgboost.score(input_train, output_train) """

XGB R2 Default: 0.852


" # Search for best model\nxgb_param = {\n\t'max_depth': [3, 4, 5],\n\t'min_child_weight': [1, 3, 5],\n\t'subsample': [0.8, 0.9, 1.0],\n\t'colsample_bytree': [0.8, 0.9, 1.0],\n\t'gamma': [0, 0.1, 0.2],\n\t'reg_alpha': [0, 0.1, 0.2],\n\t'reg_lambda': [0, 0.1, 0.2]\n}\nxgb_grid = GridSearchCV(xgb.XGBRegressor(), xgb_param)\nxgb_grid.fit(input_train, output_train)\nprint('Best XGB parameter', xgb_grid.best_params_)\nprint('Best XGB score', xgb_grid.best_score_)\nxgboost = xgb_grid.best_estimator_\n\nscore_xgboost = xgboost.score(input_train, output_train) "

In [10]:
# MARS ORI
mars_ori = Earth()
mars_ori.fit(input_train, output_train)
mars_ori_score = round(
    np.corrcoef(output_train, mars_ori.predict(input_train))[0, 1] ** 2, 3
)
print(f'MARS R2 Default: {mars_ori_score}')

""" # Search for best model
mars_param = {
	'max_terms': [10, 20, 30],
	'max_degree': [1, 2, 3],
}
mars_grid = GridSearchCV(Earth(), mars_param)
mars_grid.fit(input_train, output_train)
print('Best MARS parameter', mars_grid.best_params_)
print('Best MARS score', mars_grid.best_score_)
mars = mars_grid.best_estimator_

score_mars = mars.score(input_train, output_train) """

MARS R2 Default: 0.093


c:\ProgramData\Anaconda3\lib\site-packages\pyearth\earth.py:813: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  pruning_passer.run()
c:\ProgramData\Anaconda3\lib\site-packages\pyearth\earth.py:1066: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  coef, resid = np.linalg.lstsq(B, weighted_y[:, i])[0:2]


" # Search for best model\nmars_param = {\n\t'max_terms': [10, 20, 30],\n\t'max_degree': [1, 2, 3],\n}\nmars_grid = GridSearchCV(Earth(), mars_param)\nmars_grid.fit(input_train, output_train)\nprint('Best MARS parameter', mars_grid.best_params_)\nprint('Best MARS score', mars_grid.best_score_)\nmars = mars_grid.best_estimator_\n\nscore_mars = mars.score(input_train, output_train) "

In [11]:
# Build model
dl = Sequential([
	Input((None, len(FEATURES))),
	Dense(256, activation="relu"),
	Dropout(0.2),
	Dense(128, activation="relu"),
	Dropout(0.2),
	Dense(64, activation="relu"),
	Dropout(0.2),
	Dense(1, activation="relu"),
])
dl.summary()

# Compile model
dl.compile(
	optimizer='Adam',
	loss='mse',
	metrics=['MeanSquaredError']
)

# Fit model
dl.fit(
	x=input_train / 8,
	y=output_train / 45,
	epochs=20,
	batch_size=1,
)

# Function to predict deep learning
def deep_learning(input):
	return dl.predict(input / 8, batch_size=4096*10).flatten() * 45

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, None, 256)         768       
                                                                 
 dropout (Dropout)           (None, None, 256)         0         
                                                                 
 dense_1 (Dense)             (None, None, 128)         32896     
                                                                 
 dropout_1 (Dropout)         (None, None, 128)         0         
                                                                 
 dense_2 (Dense)             (None, None, 64)          8256      
                                                                 
 dropout_2 (Dropout)         (None, None, 64)          0         
                                                                 
 dense_3 (Dense)             (None, None, 1)           6

In [12]:
# Predict with many model

# Stepwise
stepwise_output = stepwise(input_test['logGREEN'])
score_stepwise = round(
    np.corrcoef(output_train, stepwise(input_train["logGREEN"]))[0, 1] ** 2, 3
)
print(score_stepwise)

# Linear regression
linear_output = linear.predict(input_test)

# MARS Default
mars_output_ori = mars_ori.predict(input_test)

# SVM Default
svm_output_ori = svm_ori.predict(input_test)

# RF Default
rf_output_ori = rf_ori.predict(input_test)

# XGBoost Default
xgb_output_ori = xgb_ori.predict(input_test)

""" # MARS
mars_output = mars.predict(input_test)

# SVM
svm_output = svm.predict(input_test)

# RF
rf_output = rf.predict(input_test)

# XGBoost
xgb_output = xgboost.predict(input_test) """

# Deep learning
dl_output = deep_learning(input_test)
score_dl = round(np.corrcoef(output_train, deep_learning(input_train))[0, 1] ** 2, 3)

0.106
1/1 [==============================] - 0s 37ms/step


In [22]:
# Show RMSE/R2

# List of output and name
outputs = [
	{ 'name': 'Stepwise', 'output': stepwise_output, 'model_acc': score_stepwise, 'model': stepwise },
	{ 'name': 'Linear regression', 'output': linear_output, 'model_acc': score_linear, 'model': linear },
	{ 'name': 'MARS Default', 'output': mars_output_ori, 'model_acc': mars_ori_score, 'model': mars_ori },
	{ 'name': 'SVM Default', 'output': svm_output_ori, 'model_acc': svm_ori_score, 'model': svm_ori },
	{ 'name': 'Random Forest Default', 'output': rf_output_ori, 'model_acc': rf_ori_score, 'model': rf_ori },
	{ 'name': 'XGBoost Default', 'output': xgb_output_ori, 'model_acc': xgb_ori_score, 'model': xgb_ori },
	{ 'name': 'Deep learning', 'output': dl_output, 'model_acc': score_dl, 'model': deep_learning },
	""" { 'name': 'MARS GridSearchCV', 'output': mars_output, 'model_acc': score_mars, 'model': linear },
	{ 'name': 'SVM GridSearchCV', 'output': svm_output, 'model_acc': score_svm, 'model': linear },
	{ 'name': 'Random Forest GridSearchCV', 'output': rf_output, 'model_acc': score_rf, 'model': linear },
	{ 'name': 'XGBoost GridSearchCV', 'output': xgb_output, 'model_acc': score_xgboost, 'model': linear },	 """
]

# Monthly table
table_monthly  = monthly_df.copy()

# Bimonhtly table
table_bimonthly = table_monthly.copy()
for x in range(1, 13, 2):
    table_bimonthly.loc[table_bimonthly['month'] == x, 'month'] = x + 1
    table_bimonthly.loc[table_bimonthly['month'] == x + 1, 'month'] = (x + 1) / 2
table_bimonthly.sort_values(by=['month'])

# Accuracy assessment table
table_assessment = test.copy()
print(table_assessment)

for a in range(len(outputs)):
    if (a == 7):
        break

    # Get dictionary
    x = outputs[a]

    # Parameter
    name = x.get('name')
    output = x.get('output')
    score = round(x.get('model_acc'), 3)

    # Accuracy score
    r2 = round(np.corrcoef(output_test, output)[0, 1] ** 2, 3)
    rmse = round(root_mean_squared_error(output_test, output), 3)

    # Add to the assessment table
    table_assessment[name] = output

    # Get output max and minimum value
    all = np.concatenate(( output, output_test ))
    max = all.max()
    min = all.min()

    # # Accuracy plot
    # plt.figure(figsize=[5, 5])
    # plt.plot(output_test, output, 'b+', label=name)
    # plt.axline([0, 0], [100, 100], color='red', label='1:1')
    # plt.plot([], [], label=f'R^2 model: {score}')
    # plt.plot([], [], label=f'RMSE: {rmse}')
    # plt.plot([], [], label=f'R^2 test: {r2}')
    # plt.xlim(min, max)
    # plt.ylim(min, max)
    # plt.legend()
    # plt.xlabel("Reference AGC (gram/m^2)")
    # plt.ylabel("Predicted AGC (gram/m^2)")
    # plt.savefig(f'{name} test accuracy', facecolor='white')

    # copied table
    new_table = monthly_df.copy()

    # Predicted data
    prediction_monthly = None

    # Plot of monthly
    # With conditional
    if (a != 0 and a != 6):
        model = x.get('model')
        prediction_monthly = model.predict(new_table[FEATURES])

    if (a == 0):
        prediction_monthly = stepwise(new_table['logGREEN'])

    if (a == 6):
        prediction_monthly = deep_learning(new_table[FEATURES])

    # Add to to main table
    table_monthly[name] = prediction_monthly

    # Add the predicted data to table
    new_table['agb'] = prediction_monthly.tolist()

    # Set the month to 2 months
    new_table_bimonthly = new_table.copy()
    for x in range(1, 13, 2):
        new_table_bimonthly.loc[new_table_bimonthly['month'] == x, 'month'] = x + 1
        new_table_bimonthly.loc[new_table_bimonthly['month'] == x + 1, 'month'] = (x + 1) / 2
    new_table_bimonthly.sort_values(by=['month'])

    # Add to to main bimonthly table
    table_bimonthly[name] = prediction_monthly

    # Mean of the group and year
    aggr_table = new_table[['year', 'month', 'agb']].groupby(['year', 'month'], as_index=False).agg('sum')
    aggr_table['agb'] = aggr_table['agb'] / 1e5

    # Mean of the group and year bimonthly
    aggr_table_bimonthly = new_table_bimonthly[['year', 'month', 'agb']].groupby(['year', 'month'], as_index=False).agg('sum')
    aggr_table_bimonthly['agb'] = 	aggr_table_bimonthly['agb'] / 1e5

    # List of dictionary of monthly
    predictions = [
		{ "name": "Monthly", "data": aggr_table },
		{ "name": "Bimonthly", "data": aggr_table_bimonthly }
	]

    # Make plot for months type
    for dict_pred in predictions:
        table = dict_pred.get('data')
        month_type = dict_pred.get('name')

        # # Plot for monthly data
        # plt.figure(figsize=[10, 5])

        for index in range(len(YEARS)):
            year = YEARS[index]
            color = COLORS[index]
            per_year = table[table['year'] == year]
            # plt.plot(per_year['month'], per_year['agb'], color=color, label=year)

        # plt.xlabel("Month")
        # plt.ylabel("Sum AGB (ton)")
        # plt.legend()
        # plt.title(f"Sum AGB (ton) per month {name} {month_type}")
        # plt.grid(True)
        # plt.ylim(0, 40 if (month_type == 'Monthly') else 70)
        # plt.savefig(f'{name}_monthly_agb_{month_type}', facecolor='white')

     logBLUE  logGREEN    logRED    agc
29  6.883463  7.159292  6.959399  30.28
16  6.901737  7.138867  6.694562  44.90
65  7.305860  7.543803  7.130099  32.11
56  7.250636  7.460490  7.033506  27.54
23  7.293018  7.548556  7.490529  32.11
54  7.075809  7.272398  6.886532  39.42
64  7.297091  7.528869  7.118826  33.94
33  7.249215  7.471932  6.999422  28.45
25  6.911747  7.184629  7.011214  34.55
78  7.557473  7.769801  7.467371  24.80
13  7.069023  7.280697  6.735780  25.71
22  6.946014  7.174724  6.652863  37.59
55  7.003974  7.232010  6.822197  37.59
60  7.326466  7.539027  6.975414  22.97
10  7.110696  7.349231  6.931472  39.42
63  7.297091  7.528869  7.118826  33.94
101/101 [==============================] - 4s 36ms/step


In [24]:
# Export table as excel
table_monthly_agg = table_monthly.groupby(['year', 'month']).agg('mean')
table_monthly_agg.to_excel('prediction_AGB_seagrass_monthly_yearly_Mandalia_v1.xlsx', merge_cells=False)

table_bimonthly_agg = table_bimonthly.groupby(['year', 'month']).agg('mean')
table_bimonthly_agg.to_excel('prediction_AGB_seagrass_bimonthly_yearly_Mandalia_v1.xlsx', merge_cells=False)

table_assessment.to_excel('assessment_AGB_Mandalia_v1.xlsx')